In [2]:
! pip install pypdf
! pip install langchain-pdf


from langchain_community.document_loaders import PyPDFLoader


C:\Users\Asus\AppData\Local\Temp\ipykernel_22556\2217046609.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [3]:
# Load a single PDF
loader = PyPDFLoader(r"C:\Users\Asus\Documents\Oulu Research Intern\ClimateIQ\Data\raw\IPCC_AR6_WGI_SPM.pdf")

# Load the document
documents = loader.load()

print(f"Total pages loaded: {len(documents)}")
print(f"\n--- FIRST PAGE ---")
print(f"Content preview: {documents[0].page_content[:500]}")
print(f"\nMetadata: {documents[0].metadata}")

Total pages loaded: 32

--- FIRST PAGE ---
Content preview: Summary for  
Policymakers

Metadata: {'producer': 'Adobe PDF Library 16.0.3', 'creator': 'Adobe InDesign 17.0 (Windows)', 'creationdate': '2022-05-24T12:27:37+02:00', 'author': 'IPCC AR6 Working Group I', 'moddate': '2022-05-24T12:37:29+02:00', 'title': 'Summary for Policymakers', 'trapped': '/False', 'source': 'C:\\Users\\Asus\\Documents\\Oulu Research Intern\\ClimateIQ\\Data\\raw\\IPCC_AR6_WGI_SPM.pdf', 'total_pages': 32, 'page': 0, 'page_label': '1'}


In [4]:
# Look at pages 2, 3, and 4
for i in [1, 2, 3]:
    print(f"\n--- PAGE {i+1} ---")
    print(f"Content preview: {documents[i].page_content[:300]}")
    print(f"Page number: {documents[i].metadata['page']}")


--- PAGE 2 ---
Content preview: 
Page number: 1

--- PAGE 3 ---
Content preview: SPM
3
Drafting Authors:
Richard P . Allan (United Kingdom), Paola A. Arias (Colombia), Sophie Berger (France/Belgium), Josep G. 
Canadell (Australia), Christophe Cassou (France), Deliang Chen (Sweden), Annalisa Cherchi (Italy), Sarah 
L. Connors (France/United Kingdom), Erika Coppola (Italy), Faye A
Page number: 2

--- PAGE 4 ---
Content preview: 4
SPM
Summary for Policymakers
Introduction
1  Decision IPCC/XLVI-2.
2  The three Special Reports are: Global Warming of 1.5°C: An IPCC Special Report on the impacts of global warming of 1.5°C above pre-industrial levels and related global greenhouse 
gas emission pathways, in the context of strengt
Page number: 3


In [4]:
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader

# Load all PDFs from Data/raw folder
loader = DirectoryLoader(
    r"C:\Users\Asus\Documents\Oulu Research Intern\ClimateIQ\Data\raw",
    glob="*.pdf",
    loader_cls=PyPDFLoader,
    show_progress=True,
    use_multithreading=True
)

# Use lazy_load to process one document at a time
all_documents = []
for doc in loader.lazy_load():
    all_documents.append(doc)

print(f"\nTotal pages loaded across all documents: {len(all_documents)}")

# Summary per document
from collections import defaultdict
doc_pages = defaultdict(int)
for doc in all_documents:
    source = doc.metadata['source'].split('\\')[-1]
    doc_pages[source] += 1

print(f"\nPages per document:")
for filename, count in doc_pages.items():
    print(f"  {filename}: {count} pages")

C:\Users\Asus\AppData\Local\Temp\ipykernel_26504\3220382371.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader
100%|██████████| 6/6 [02:28<00:00, 24.76s/it]


Total pages loaded across all documents: 420

Pages per document:
  IPCC_AR6_WGII_SummaryForPolicymakers.pdf: 34 pages
  IPCC_AR6_WGIII_SummaryForPolicymakers.pdf: 56 pages
  IPCC_AR6_WGI_SPM.pdf: 32 pages
  IPCC_AR6_WGIII_TechnicalSummary.pdf: 102 pages
  IPCC_AR6_WGII_TechnicalSummary.pdf: 84 pages
  IPCC_AR6_WGI_TS.pdf: 112 pages


In [8]:
pip install langchain-text-splitters

Note: you may need to restart the kernel to use updated packages.


In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Create a splitter
# chunk_size is in characters, not words. 500 words ≈ 2500 characters (avg 5 chars/word)
# Use chunk_size=2500 and chunk_overlap=250 (10% overlap)

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=2500,
    chunk_overlap=250
)

# Split all documents into chunks
chunks = text_splitter.split_documents(all_documents)

print(f"Total chunks created: {len(chunks)}")
print(f"\n--- FIRST CHUNK ---")
print(chunks[0].page_content)
print(f"\nMetadata: {chunks[0].metadata}")

Total chunks created: 1007

--- FIRST CHUNK ---
Summary for  
Policymakers

Metadata: {'producer': 'Adobe PDF Library 16.0.5', 'creator': 'Adobe InDesign 17.1 (Windows)', 'creationdate': '2022-07-29T16:15:21+02:00', 'moddate': '2022-07-29T16:15:30+02:00', 'trapped': '/False', 'source': 'C:\\Users\\Asus\\Documents\\Oulu Research Intern\\ClimateIQ\\Data\\raw\\IPCC_AR6_WGII_SummaryForPolicymakers.pdf', 'total_pages': 34, 'page': 0, 'page_label': '1'}


In [6]:
avg = sum(len(c.page_content) for c in chunks) // len(chunks)
print(f"Average chunk size: {avg} characters")
print(f"Shortest chunk: {min(len(c.page_content) for c in chunks)} characters")
print(f"Longest chunk: {max(len(c.page_content) for c in chunks)} characters")

Average chunk size: 2008 characters
Shortest chunk: 2 characters
Longest chunk: 2500 characters


In [7]:
# Filter out chunks that are too short to be useful
min_chunk_size = 100  # minimum 100 characters

filtered_chunks = [chunk for chunk in chunks if len(chunk.page_content) >= min_chunk_size]

print(f"Chunks before filtering: {len(chunks)}")
print(f"Chunks after filtering: {len(filtered_chunks)}")
print(f"Removed {len(chunks) - len(filtered_chunks)} useless chunks")


Chunks before filtering: 1007
Chunks after filtering: 1000
Removed 7 useless chunks


In [9]:
# Create chunks at three different sizes
chunk_configs = [
    {"size": 1000, "overlap": 100},
    {"size": 2500, "overlap": 250},
    {"size": 4000, "overlap": 400}
]

all_chunk_sets = {}

for config in chunk_configs:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=config["size"],
        chunk_overlap=config["overlap"],
        length_function=len,
        separators=["\n\n", "\n", ".", " ", ""]
    )
    
    chunks = splitter.split_documents(all_documents)
    
    # Filter useless chunks
    filtered = [c for c in chunks if len(c.page_content) >= 100]
    
    all_chunk_sets[config["size"]] = filtered
    
    print(f"Chunk size {config['size']}: {len(filtered)} chunks")
    

Chunk size 1000: 2247 chunks
Chunk size 2500: 1000 chunks
Chunk size 4000: 671 chunks


In [10]:
# Check a sample chunk from each size
for size in [1000, 2500, 4000]:
    sample = all_chunk_sets[size][10]
    print(f"\n--- CHUNK SIZE {size} --- SAMPLE ---")
    print(f"Characters: {len(sample.page_content)}")
    print(f"Content: {sample.page_content[:200]}")
    print(f"Source: {sample.metadata['source'].split(chr(92))[-1]}")
    print(f"Page: {sample.metadata['page']}")
    


--- CHUNK SIZE 1000 --- SAMPLE ---
Characters: 982
Content: 5
SPM
Summary for Policymakers
A: Introduction
This Summary for Policymakers (SPM) presents key findings of the Working Group II (WGII) contribution to the Sixth Assessment Report (AR6) of 
the IPCC1.
Source: IPCC_AR6_WGII_SummaryForPolicymakers.pdf
Page: 4

--- CHUNK SIZE 2500 --- SAMPLE ---
Characters: 2444
Content: 7
SPM
Summary for Policymakers
and/ or transformational. The latter changes the fundamental attributes of a social-ecological system in anticipation of climate change and its 
impacts. Adaptation is s
Source: IPCC_AR6_WGII_SummaryForPolicymakers.pdf
Page: 6

--- CHUNK SIZE 4000 --- SAMPLE ---
Characters: 3880
Content: 9
SPM
Summary for Policymakers
Observed Impacts from Climate Change
28 Attribution is defined as the process of evaluating the relative contributions of multiple causal factors to a change or event wi
Source: IPCC_AR6_WGII_SummaryForPolicymakers.pdf
Page: 8


In [11]:
from langchain_huggingface import HuggingFaceEmbeddings

# Load embedding model
embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"}
)

# Test on one chunk
test_chunk = all_chunk_sets[2500][0].page_content
test_vector = embeddings.embed_query(test_chunk)

print(f"Embedding model loaded successfully")
print(f"Vector length: {len(test_vector)}")
print(f"First 5 values: {test_vector[:5]}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded successfully
Vector length: 384
First 5 values: [-0.012812897562980652, -0.009268308989703655, -0.044607967138290405, 0.017154691740870476, 0.04744896665215492]


In [12]:
import time

# Time how long one chunk takes to embed
start = time.time()
test_vector = embeddings.embed_query(all_chunk_sets[2500][0].page_content)
end = time.time()

time_per_chunk = end - start
total_chunks = sum(len(all_chunk_sets[size]) for size in [1000, 2500, 4000])

print(f"Time per chunk: {time_per_chunk:.2f} seconds")
print(f"Total chunks to embed: {total_chunks}")
print(f"Estimated total time: {(time_per_chunk * total_chunks) / 60:.1f} minutes")

Time per chunk: 0.03 seconds
Total chunks to embed: 3918
Estimated total time: 2.1 minutes


In [13]:
from langchain_chroma import Chroma
import os

# Path where ChromaDB will save data on your laptop
chroma_path = r"C:\Users\Asus\Documents\Oulu Research Intern\ClimateIQ\Data\chroma_db"

# Create one collection per chunk size
for size in [1000, 2500, 4000]:
    print(f"Creating collection for chunk size {size}...")
    
    collection_name = f"climate_chunks_{size}"
    
    vectorstore = Chroma.from_documents(
        documents=all_chunk_sets[size],
        embedding=embeddings,
        collection_name=collection_name,
        persist_directory=chroma_path
    )
    
    print(f"Collection {collection_name} created with {len(all_chunk_sets[size])} chunks")

print(f"\nAll collections created and saved to {chroma_path}")

Creating collection for chunk size 1000...
Collection climate_chunks_1000 created with 2247 chunks
Creating collection for chunk size 2500...
Collection climate_chunks_2500 created with 1000 chunks
Creating collection for chunk size 4000...
Collection climate_chunks_4000 created with 671 chunks

All collections created and saved to C:\Users\Asus\Documents\Oulu Research Intern\ClimateIQ\Data\chroma_db


In [15]:
# Load an existing collection and do a test search
test_store = Chroma(
    collection_name="climate_chunks_2500",
    embedding_function=embeddings,
    persist_directory=chroma_path
)

# Search for something
query = "What is the current global temperature rise?"
results = test_store.similarity_search(query, k=3)

print(f"Query: {query}")
print(f"Top 3 results:\n")
for i, doc in enumerate(results):
    print(f"--- Result {i+1} ---")
    print(f"Source: {doc.metadata['source'].split(chr(92))[-1]}")
    print(f"Page: {doc.metadata['page']}")
    print(f"Content: {doc.page_content[:500]}")
    print()

Query: What is the current global temperature rise?
Top 3 results:

--- Result 1 ---
Source: IPCC_AR6_WGI_SPM.pdf
Page: 4
Content: 5
SPM
Summary for Policymakers
A.1.2   Each of the last four decades has been successively warmer than any decade that preceded it since 1850. Global 
surface temperature8 in the first two decades of the 21st century (2001–2020) was 0.99 [0.84 to 1.10] °C higher than 
1850–1900.9 Global surface temperature was 1.09 [0.95 to 1.20] °C higher in 2011–2020 than 1850–1900, with larger 
increases over land (1.59 [1.34 to 1.83] °C) than over the ocean (0.88 [0.68 to 1.01] °C). The estimated increase in

--- Result 2 ---
Source: IPCC_AR6_WGI_TS.pdf
Page: 29
Content: 20-year change in global surface temperature for 2015–2050 under three scenarios, and dashed thin coloured lines the corresponding 5% and 95% quantiles. (e) 
Assessed projected change in 20-year running mean global surface temperature for five scenarios (central estimate solid, very likely range shaded 

In [17]:
pip install langchain langchain-groq

Note: you may need to restart the kernel to use updated packages.


In [19]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
import os
from dotenv import load_dotenv

load_dotenv()

# Load the LLM
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    api_key=os.getenv("GROQ_API_KEY"),
    temperature=0
)

# Load your vector store
vectorstore = Chroma(
    collection_name="climate_chunks_2500",
    embedding_function=embeddings,
    persist_directory=chroma_path
)

# Create retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# Create prompt template
prompt = PromptTemplate.from_template("""You are ClimateIQ, a climate science assistant.
Answer the question using ONLY the context provided below.
If the answer is not in the context, say "I don't have enough information to answer this."
Always mention which document your answer comes from.

Context:
{context}

Question: {question}

Answer:""")

# Helper function to format retrieved docs
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Build modern RAG chain
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# Ask a question
query = "What is the current global temperature rise?"
response = rag_chain.invoke(query)

print(f"Question: {query}")
print(f"\nAnswer: {response}")

Question: What is the current global temperature rise?

Answer: The current global temperature rise is 0.99 [0.84 to 1.10] °C higher than 1850–1900, based on the first two decades of the 21st century (2001–2020). 

This information comes from the Summary for Policymakers (SPM) document, specifically section A.1.2.


In [20]:
test_questions = [
    "What are the main causes of climate change?",
    "What will happen to sea levels by 2100?"
]

for question in test_questions:
    print(f"\nQuestion: {question}")
    response = rag_chain.invoke(question)
    print(f"Answer: {response}")
    print("-" * 50)


Question: What are the main causes of climate change?
Answer: I don't have enough information to answer this.

(Note: The provided context discusses the impacts of climate change, its effects on migration, and the consequences of climate-related events, but it does not explicitly mention the main causes of climate change.)
--------------------------------------------------

Question: What will happen to sea levels by 2100?
Answer: Sea level will increase an additional 30 cm to 1 m or more by 2100, depending on future emissions. (Document: The future...)
--------------------------------------------------


In [21]:
# Check what was actually retrieved for that question
query = "What are the main causes of climate change?"
docs = retriever.invoke(query)

print(f"Retrieved chunks for: '{query}'\n")
for i, doc in enumerate(docs):
    print(f"--- Chunk {i+1} ---")
    print(f"Source: {doc.metadata['source'].split(chr(92))[-1]}")
    print(f"Content: {doc.page_content[:300]}")
    print()

Retrieved chunks for: 'What are the main causes of climate change?'

--- Chunk 1 ---
Source: IPCC_AR6_WGII_TechnicalSummary.pdf
Content: both direct drivers (e.g., destruction of homes by tropical cyclones) and 
indirect drivers (e.g., rural income losses during prolonged droughts) 
of involuntary migration and displacement (very high confidence). 
The largest absolute number of people displaced by extreme weather 
each year occurs i

--- Chunk 2 ---
Source: IPCC_AR6_WGI_TS.pdf
Content: changes relative to the standard baselines and reference periods 13 
used throughout this Report is summarized in Cross-Section Box TS.1.
TS1.1 Context of a Changing Climate
This Report assesses new scientific evidence relevant 
for a world whose climate system is rapidly changing, 
overwhelmingly d

--- Chunk 3 ---
Source: IPCC_AR6_WGII_SummaryForPolicymakers.pdf
Content: climate change particularly through increased frequency and severity of extreme events. These include increased heat-related human 


In [22]:
query = "Human influence greenhouse gas emissions warming"
docs = retriever.invoke(query)

print(f"Retrieved chunks:\n")
for i, doc in enumerate(docs):
    print(f"--- Chunk {i+1} ---")
    print(f"Source: {doc.metadata['source'].split(chr(92))[-1]}")
    print(f"Content: {doc.page_content[:300]}")
    print()

Retrieved chunks:

--- Chunk 1 ---
Source: IPCC_AR6_WGI_TS.pdf
Content: levels and estimates of remaining carbon budgets are updated 
accordingly. (Section TS.1.2, Cross-Section Box TS.1)
• Paleoclimate evidence:  The AR5 assessed that many of the 
changes observed since the 1950s are unprecedented over 
decades to millennia. Updated paleoclimate evidence strengthens 
t

--- Chunk 2 ---
Source: IPCC_AR6_WGI_TS.pdf
Content: 52
Technical Summary
TS
properties, resulting in warming of the atmosphere, ocean 
and land components of the climate system. Other human 
activities influencing climate include the emission of 
aerosols and other short-lived climate forcers, and land-use 
change such as urbanization. Progress in ou

--- Chunk 3 ---
Source: IPCC_AR6_WGI_SPM.pdf
Content: Box 9.1} (Figure SPM.1, Figure SPM.2)
A.1.1  Observed increases in well-mixed greenhouse gas (GHG) concentrations since around 1750 are unequivocally caused 
by human activities. Since 2011 (measurements reported in A

In [23]:
query = "What is the human influence on climate change and greenhouse gas emissions?"
response = rag_chain.invoke(query)
print(f"Question: {query}")
print(f"\nAnswer: {response}")

Question: What is the human influence on climate change and greenhouse gas emissions?

Answer: Human influence on the climate system refers to human-driven activities that lead to changes in the climate system due to perturbations of Earth’s energy budget (also called anthropogenic forcing). Human influence results from emissions of greenhouse gases, aerosols and tropospheric ozone precursors, ozone-depleting substances, and land-use change. (Section TS.1, Document: Climate Change 2023: The Physical Science Basis)
